# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² (FAIR2) dataset: **"Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells"** using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and follows the FAIR (Findable, Accessible, Interoperable, Reusable) principles.


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version} | License: {metadata.license}")

## 2. Data Overview
Review available record sets, their fields, and corresponding `@id`s. All references use the entity `@id` as required by the Croissant schema.


In [ ]:
# List all record sets and their fields by @id
print("Available Record Sets:")

record_set_ids = []
for rs in metadata.record_sets():
    print(f"- RecordSet name: {rs.name} | @id: {rs.id}")
    record_set_ids.append(rs.id)
    field_strs = []
    for f in rs.fields():
        field_strs.append(f"    - Field name: {f.name} | @id: {f.id} | DataType: {f.data_type}")
    if field_strs:
        print("      Fields:")
        print("\n".join(field_strs))
print("\nTotal record sets:", len(record_set_ids))

## 3. Data Extraction
Load one or more record sets into pandas DataFrames using their Croissant `@id`. Use the field `@id` for column lookups.


In [ ]:
# For demonstration, extract all record sets (usually you select the main one).
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"Warning: No records found for record set {record_set_id}")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"DataFrame loaded for RecordSet @id: {record_set_id} -- shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")

# For the rest of the notebook, select the main record set by ID.
# If only one record set, pick it. If multiple, you may pick the most detailed dataset.
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Using main_record_set_id: {main_record_set_id}")
else:
    main_record_set_id = None  # No record sets found

# Display sample records for the main record set
if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    display(df.head())
else:
    print("No main record set with data to display.")

## 4. Exploratory Data Analysis (EDA)
We'll demonstrate numeric operations (filtering, normalization, grouping) using the available fields. Use the field's `@id` (not just name) for every operation.


In [ ]:
# Identify a numeric field by dataType (e.g., float/int/number). We'll use the first one found.
numeric_field_id = None
group_field_id = None

for rs in metadata.record_sets():
    if rs.id != main_record_set_id:
        continue
    for f in rs.fields():
        if f.data_type in ['schema:Number', 'schema:Integer', 'schema:Float']:
            numeric_field_id = f.id
            break
    # Try to find a non-numeric (categorical) for grouping
    for f in rs.fields():
        if f.data_type == 'schema:Text':
            group_field_id = f.id
            break

if not numeric_field_id:
    print("No numeric field detected for main record set. Please adjust selection.")
else:
    print(f"Using numeric_field_id: {numeric_field_id}")

    df = dataframes[main_record_set_id]
    # Convert values to numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    
    # Filter: Example threshold
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean): {filtered_df.shape[0]} records")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a text field if available
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} by {group_field_id} (first 5 groups):")
        display(grouped_df.head())
    else:
        print("No groupable text field found.")

## 5. Visualization
Plot the distribution of the selected numeric field and compare groups if available.


In [ ]:
import matplotlib.pyplot as plt

if main_record_set_id and numeric_field_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    plt.figure(figsize=(8, 4))
    df[numeric_field_id].dropna().hist(bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 4))
        df.boxplot(column=numeric_field_id, by=group_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.suptitle('')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("Visualization skipped; no suitable numeric or group field found.")

## 6. Conclusion
In this notebook, we successfully loaded, explored, and visualized the FAIR² mass spectrometry dataset using Croissant's rich schema and unique `@id` referencing. The demonstration included programmatic discovery of record sets and fields, field selection by data type, numeric normalization, grouping, and visualization—all powered by the `mlcroissant` library's API. Explore the full schema and all data fields to tailor analyses to your research needs!
